In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from pycaret.regression import *
from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("C:\\Users\\ripa_\\Desktop\\Programing\\IndyCar_Project\\datasets\\IndyCar_dataset_v16.csv")

In [3]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values("EventDate")

In [4]:
print(df[["DriverID", "NormalizedPositionFinish", "DRFAvg"]].groupby("DriverID").head(3).head(30))

    DriverID  NormalizedPositionFinish  DRFAvg
0       3608                      0.52     NaN
25      4401                      0.88     NaN
24      4021                      0.80     NaN
23      3675                      0.56     NaN
22      4276                      0.20     NaN
21      4236                      1.00     NaN
20      4215                      0.40     NaN
19      4144                      0.32     NaN
17      3682                      0.36     NaN
16      3668                      0.44     NaN
15      3636                      0.48     NaN
14      3628                      0.04     NaN
13      3625                      0.76     NaN
18      3811                      0.84     NaN
11      4216                      0.12     NaN
12      4407                      0.64     NaN
2       3620                      0.68     NaN
3       3622                      0.00     NaN
4       3645                      0.08     NaN
5       3648                      0.96     NaN
1       3616 

In [5]:
df.head()

,DriverName,DriverID,Rookie,DRFAvg,DTAvg,DTTAvg,DNFRate,TDNFRate,DriverElo,DriverTElo,...,EventDate,EventDateFormatted,EventID,Era,EraID,Status,StatusID,FieldSize,PositionFinish,NormalizedPositionFinish
0,Marco Andretti,3608,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,2012-03-25,"Sunday, March 25, 2012",2380,DW12 Era 2012-2017,0,Running,0,26,14,0.52
25,Katherine Legge,4401,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,2012-03-25,"Sunday, March 25, 2012",2380,DW12 Era 2012-2017,0,DNF,1,26,23,0.88
24,Sebastien Bourdais,4021,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,2012-03-25,"Sunday, March 25, 2012",2380,DW12 Era 2012-2017,0,DNF,1,26,21,0.80
23,Alex Tagliani,3675,0,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,2012-03-25,"Sunday, March 25, 2012",2380,DW12 Era 2012-2017,0,Running,0,26,15,0.56
22,Simon Pagenaud,4276,1,NaN,NaN,NaN,NaN,NaN,1500.0,1500.0,...,2012-03-25,"Sunday, March 25, 2012",2380,DW12 Era 2012-2017,0,Running,0,26,6,0.20


In [ ]:
#Pre-Qualy Non-ID Model
drop_cols = [
    "DriverName", "DriverID", "PositionStart", "TeamName", "TeamID", "CarEngine", "EngineID", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [ ]:
#Pre-Qualy ID Model
drop_cols = [
    "DriverName", "PositionStart", "TeamName", "CarEngine", "EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

In [6]:
#Post-Qualy Model
drop_cols = [
    "DriverName", "TeamName", "CarEngine","EventName", "Track", "EventTrackType",
    "EventDate", "EventDateFormatted", "EventID", "Era",
    "Status", "StatusID", "PositionFinish"
]

cutoff = df["EventDate"].quantile(0.95)
data = df[df["EventDate"] < cutoff].drop(columns=drop_cols)
data_unseen = df[df["EventDate"] >= cutoff].drop(columns=drop_cols)

print(data.corr(numeric_only=True)["NormalizedPositionFinish"].sort_values())

DriverElo                  -3.945180e-01
DriverTTElo                -3.641252e-01
TeamElo                    -3.267063e-01
TeamTElo                   -2.431758e-01
DriverTElo                 -2.405731e-01
TeamID                     -1.258151e-01
EngineTTElo                -7.331906e-02
EngineElo                  -6.209779e-02
EngineTElo                 -4.255228e-02
TrackID                    -2.567460e-03
EraID                      -1.571381e-15
FieldSize                  -1.007089e-15
EventTrackTypeID            4.008219e-17
EngineID                    2.255965e-02
TeamDNFRate                 2.730076e-02
TDNFRate                    6.637972e-02
DriverID                    9.888619e-02
DNFRate                     1.671002e-01
Rookie                      1.673821e-01
DTAvg                       2.581422e-01
TTP                         3.049000e-01
TRP                         3.128037e-01
TeamRitmo                   3.364790e-01
DRFAvg                      3.607147e-01
DTTAvg          

In [7]:
df = df.drop(columns=drop_cols)

In [8]:
print(df.columns.tolist())

['DriverID', 'Rookie', 'DRFAvg', 'DTAvg', 'DTTAvg', 'DNFRate', 'TDNFRate', 'DriverElo', 'DriverTElo', 'DriverTTElo', 'DriverRitmo', 'PositionStart', 'TeamID', 'TRP', 'TTP', 'TeamDNFRate', 'TeamElo', 'TeamTElo', 'TeamRitmo', 'EngineID', 'EngineElo', 'EngineTElo', 'EngineTTElo', 'TrackID', 'EventTrackTypeID', 'EraID', 'FieldSize', 'NormalizedPositionFinish']


In [9]:
exp = setup(
    data=data, 
    target="NormalizedPositionFinish", 
    session_id=123, 
    fold_strategy="timeseries",
    data_split_shuffle=False,
    fold_shuffle=False
)

,Description,Value
0,Session id,123
1,Target,NormalizedPositionFinish
2,Target type,Regression
3,Original data shape,"(5344, 28)"
4,Transformed data shape,"(5344, 28)"
5,Transformed train set shape,"(3740, 28)"
6,Transformed test set shape,"(1604, 28)"
7,Numeric features,27
8,Rows with missing values,28.8%
9,Preprocess,True


In [ ]:
compare_models()

In [10]:
et = create_model('et')
et_tune = tune_model(et)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2348,0.0781,0.2796,0.1260,0.1920,1.0278
1,0.2533,0.0894,0.2990,0.0234,0.2074,1.1723
2,0.2393,0.0805,0.2837,0.1144,0.1957,1.0468
3,0.2353,0.0782,0.2797,0.1341,0.1923,1.0213
4,0.2389,0.0803,0.2833,0.1289,0.1944,1.0259
5,0.2284,0.0755,0.2748,0.1547,0.1902,1.0402
6,0.2358,0.0803,0.2833,0.1103,0.1936,1.0265
7,0.2349,0.0767,0.2769,0.1657,0.1889,0.9725
8,0.2279,0.0715,0.2674,0.2092,0.1825,0.9703


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2363,0.0775,0.2784,0.1332,0.1900,1.0120
1,0.2437,0.0819,0.2862,0.1050,0.1958,1.0276
2,0.2388,0.0802,0.2831,0.1179,0.1940,1.0374
3,0.2298,0.0755,0.2747,0.1644,0.1867,0.9347
4,0.2346,0.0769,0.2773,0.1651,0.1882,0.9372
5,0.2207,0.0707,0.2658,0.2090,0.1820,0.9419
6,0.2269,0.0747,0.2733,0.1721,0.1851,0.9353
7,0.2344,0.0769,0.2773,0.1637,0.1879,0.9418
8,0.2237,0.0707,0.2659,0.2179,0.1800,0.9049


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
#predict_model(et_tune);
predict_model(et);
newpred1 = predict_model(et, data=data_unseen)
#newpred1 = predict_model(et_tune, data=data_unseen)

In [11]:
rf = create_model('rf')
rf_tune = tune_model(rf)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2371,0.0793,0.2816,0.1135,0.1933,1.0478
1,0.2574,0.0932,0.3053,-0.0183,0.2116,1.2074
2,0.2402,0.0808,0.2843,0.1106,0.1965,1.1001
3,0.2374,0.0796,0.2821,0.1186,0.1948,1.0553
4,0.2480,0.0834,0.2887,0.0953,0.1983,1.0767
5,0.2275,0.0742,0.2725,0.1689,0.1888,1.0251
6,0.2301,0.0768,0.2772,0.1481,0.1889,0.9901
7,0.2358,0.0785,0.2801,0.1464,0.1913,0.9994
8,0.2234,0.0701,0.2649,0.2240,0.1807,0.9425


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2392,0.0782,0.2797,0.1251,0.1923,1.0710
1,0.2467,0.0849,0.2914,0.0725,0.2017,1.1280
2,0.2380,0.0799,0.2827,0.1206,0.1948,1.0602
3,0.2321,0.0756,0.2749,0.1635,0.1883,0.9858
4,0.2369,0.0770,0.2775,0.1640,0.1897,0.9935
5,0.2219,0.0704,0.2653,0.2120,0.1828,0.9724
6,0.2283,0.0753,0.2745,0.1649,0.1867,0.9657
7,0.2327,0.0755,0.2747,0.1791,0.1870,0.9588
8,0.2212,0.0687,0.2620,0.2404,0.1781,0.9125


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [12]:
predict_model(rf_tune);
#predict_model(rf);
newpred2 = predict_model(rf_tune, data=data_unseen)
#newpred2 = predict_model(rf, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2125,0.0649,0.2548,0.2765,0.1736,0.9310


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Random Forest Regressor,0.2128,0.0665,0.2578,0.2607,0.1750,0.9214


In [13]:
gbr = create_model('gbr')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2428,0.0861,0.2935,0.0367,0.1995,1.0038
1,0.2694,0.1041,0.3226,-0.1373,0.2221,1.2009
2,0.2460,0.0870,0.2950,0.0424,0.2035,1.1110
3,0.2406,0.0828,0.2878,0.0831,0.1975,1.0453
4,0.2423,0.0823,0.2869,0.1064,0.1959,1.0529
5,0.2275,0.0765,0.2766,0.1435,0.1909,0.9912
6,0.2238,0.0754,0.2747,0.1636,0.1860,0.9411
7,0.2400,0.0823,0.2869,0.1049,0.1962,0.9921
8,0.2208,0.0693,0.2632,0.2337,0.1779,0.8958


In [14]:
gbr_tune = tune_model(gbr)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2384,0.0779,0.2791,0.1287,0.1917,1.0542
1,0.2536,0.0885,0.2974,0.0333,0.2059,1.1742
2,0.2425,0.0811,0.2848,0.1071,0.1971,1.1133
3,0.2386,0.0785,0.2802,0.1307,0.1929,1.0358
4,0.2440,0.0796,0.2821,0.1360,0.1941,1.0501
5,0.2288,0.0741,0.2723,0.1700,0.1883,1.0212
6,0.2291,0.0741,0.2722,0.1784,0.1861,0.9990
7,0.2361,0.0765,0.2766,0.1681,0.1891,1.0006
8,0.2266,0.0707,0.2659,0.2179,0.1816,0.9726


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(gbr);
#predict_model(gbr_tune);

In [ ]:
#newpred3 = predict_model(gbr_tune, data=data_unseen)
newpred3 = predict_model(gbr, data=data_unseen)

In [15]:
cat = create_model('catboost')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2390,0.0805,0.2837,0.0996,0.1937,1.0187
1,0.2698,0.1043,0.3229,-0.1396,0.2233,1.2778
2,0.2487,0.0896,0.2994,0.0134,0.2072,1.1173
3,0.2436,0.0856,0.2925,0.0524,0.2017,1.0632
4,0.2518,0.0889,0.2981,0.0356,0.2023,1.0345
5,0.2274,0.0773,0.2780,0.1347,0.1913,1.0009
6,0.2309,0.0804,0.2836,0.1087,0.1920,0.9670
7,0.2355,0.0803,0.2833,0.1267,0.1929,0.9332
8,0.2273,0.0738,0.2717,0.1832,0.1836,0.9120


In [16]:
cat_tune = tune_model(cat)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2365,0.0795,0.2819,0.1111,0.1925,1.0123
1,0.2460,0.0848,0.2911,0.0739,0.2013,1.1028
2,0.2343,0.0793,0.2815,0.1277,0.1944,1.0441
3,0.2326,0.0771,0.2777,0.1464,0.1907,0.9895
4,0.2361,0.0784,0.2800,0.1491,0.1905,0.9719
5,0.2204,0.0705,0.2656,0.2106,0.1822,0.9422
6,0.2262,0.0758,0.2753,0.1600,0.1862,0.9401
7,0.2325,0.0775,0.2784,0.1571,0.1886,0.9302
8,0.2148,0.0664,0.2577,0.2652,0.1746,0.8732


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(cat_tune);
#predict_model(cat);

In [ ]:
newpred5 = predict_model(cat_tune, data=data_unseen)
#newpred5 = predict_model(cat, data=data_unseen)

In [17]:
lgbm = create_model('lightgbm')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2532,0.0923,0.3038,-0.0323,0.2078,1.1348
1,0.2783,0.1118,0.3344,-0.2219,0.2307,1.3056
2,0.2480,0.0929,0.3047,-0.0219,0.2102,1.1034
3,0.2478,0.0890,0.2984,0.0142,0.2052,1.0843
4,0.2426,0.0860,0.2933,0.0662,0.1993,0.9714
5,0.2340,0.0811,0.2848,0.0919,0.1955,0.9836
6,0.2234,0.0753,0.2745,0.1649,0.1847,0.8562
7,0.2353,0.0815,0.2855,0.1135,0.1945,0.9337
8,0.2300,0.0753,0.2744,0.1673,0.1842,0.8792


In [18]:
lgbm_tune = tune_model(lgbm)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2410,0.0790,0.2811,0.1161,0.1930,1.0553
1,0.2448,0.0830,0.2881,0.0928,0.1979,1.0513
2,0.2382,0.0815,0.2855,0.1029,0.1963,1.0498
3,0.2284,0.0745,0.2730,0.1750,0.1861,0.9401
4,0.2352,0.0767,0.2770,0.1670,0.1886,0.9674
5,0.2198,0.0700,0.2645,0.2167,0.1814,0.9355
6,0.2252,0.0743,0.2725,0.1768,0.1852,0.9541
7,0.2316,0.0761,0.2759,0.1719,0.1877,0.9452
8,0.2185,0.0677,0.2602,0.2511,0.1765,0.9000


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [ ]:
predict_model(lgbm_tune);
#predict_model(lgbm);

In [ ]:
newpred5 = predict_model(lgbm_tune, data=data_unseen)
#newpred5 = predict_model(lgbm, data=data_unseen)

In [19]:
br = create_model('br')

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2364,0.0808,0.2843,0.0961,0.1929,0.9780
1,0.2446,0.0847,0.2911,0.0740,0.1993,1.0181
2,0.2322,0.0811,0.2847,0.1080,0.1957,0.9914
3,0.2270,0.0741,0.2721,0.1800,0.1852,0.9130
4,0.2291,0.0763,0.2761,0.1724,0.1869,0.8970
5,0.2159,0.0703,0.2652,0.2126,0.1807,0.8779
6,0.2190,0.0734,0.2709,0.1863,0.1828,0.8687
7,0.2292,0.0765,0.2766,0.1677,0.1876,0.9098
8,0.2106,0.0651,0.2552,0.2797,0.1728,0.8386


In [20]:
br_tune = tune_model(br)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2356,0.0806,0.2840,0.0982,0.1928,0.9717
1,0.2453,0.0855,0.2924,0.0655,0.2002,1.0170
2,0.2313,0.0811,0.2849,0.1070,0.1958,0.9826
3,0.2262,0.0739,0.2718,0.1820,0.1850,0.9098
4,0.2284,0.0759,0.2755,0.1759,0.1866,0.8967
5,0.2162,0.0706,0.2657,0.2096,0.1810,0.8765
6,0.2186,0.0734,0.2709,0.1867,0.1827,0.8644
7,0.2289,0.0764,0.2764,0.1689,0.1875,0.9116
8,0.2100,0.0649,0.2547,0.2824,0.1724,0.8329


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [22]:
predict_model(br_tune);
#predict_model(br);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Bayesian Ridge,0.2026,0.0642,0.2534,0.2844,0.1707,0.8039


In [21]:
newpred9 = predict_model(br_tune, data=data_unseen)
#newpred9 = predict_model(br, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Bayesian Ridge,0.2053,0.0667,0.2583,0.2582,0.1729,0.7968


In [23]:
blend1 = blend_models([rf_tune, lgbm_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2387,0.0778,0.2790,0.1295,0.1918,1.0607
1,0.2447,0.0831,0.2883,0.0919,0.1989,1.0873
2,0.2379,0.0804,0.2836,0.1149,0.1952,1.0545
3,0.2299,0.0748,0.2734,0.1722,0.1869,0.9622
4,0.2358,0.0767,0.2769,0.1679,0.1889,0.9797
5,0.2207,0.0699,0.2645,0.2171,0.1818,0.9536
6,0.2265,0.0746,0.2732,0.1728,0.1857,0.9593
7,0.2319,0.0756,0.2750,0.1773,0.1872,0.9513
8,0.2196,0.0680,0.2608,0.2478,0.1770,0.9056


In [24]:
blend1_tune = tune_model(blend1)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2388,0.0779,0.2791,0.1287,0.1919,1.0597
1,0.2446,0.0830,0.2881,0.0932,0.1986,1.0816
2,0.2379,0.0805,0.2838,0.1136,0.1953,1.0537
3,0.2297,0.0747,0.2733,0.1730,0.1867,0.9590
4,0.2356,0.0766,0.2769,0.1681,0.1888,0.9779
5,0.2205,0.0699,0.2644,0.2174,0.1817,0.9510
6,0.2262,0.0745,0.2730,0.1736,0.1856,0.9585
7,0.2318,0.0757,0.2751,0.1767,0.1872,0.9504
8,0.2194,0.0679,0.2606,0.2485,0.1769,0.9047


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [48]:
predict_model(blend1_tune);
#predict_model(blend1);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2090,0.0636,0.2522,0.2912,0.1715,0.9012


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [50]:
newpred5 = predict_model(blend1_tune, data=data_unseen)
#newpred5 = predict_model(blend1, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2088,0.0652,0.2553,0.2750,0.1729,0.8929


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [25]:
blend2 = blend_models([rf_tune, lgbm_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2373,0.0777,0.2787,0.1314,0.1911,1.0429
1,0.2444,0.0831,0.2882,0.0922,0.1990,1.0910
2,0.2360,0.0795,0.2820,0.1246,0.1944,1.0496
3,0.2302,0.0750,0.2739,0.1694,0.1875,0.9698
4,0.2356,0.0768,0.2772,0.1660,0.1889,0.9764
5,0.2204,0.0699,0.2643,0.2179,0.1816,0.9494
6,0.2262,0.0746,0.2732,0.1727,0.1855,0.9524
7,0.2319,0.0760,0.2757,0.1730,0.1874,0.9439
8,0.2175,0.0673,0.2593,0.2560,0.1759,0.8938


In [26]:
blend2_tune = tune_model(blend2)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2378,0.0777,0.2787,0.1314,0.1913,1.0490
1,0.2443,0.0830,0.2880,0.0934,0.1988,1.0868
2,0.2367,0.0799,0.2826,0.1210,0.1947,1.0511
3,0.2299,0.0748,0.2735,0.1716,0.1871,0.9653
4,0.2355,0.0767,0.2770,0.1673,0.1888,0.9768
5,0.2204,0.0699,0.2643,0.2180,0.1816,0.9498
6,0.2261,0.0745,0.2730,0.1736,0.1855,0.9546
7,0.2318,0.0759,0.2755,0.1747,0.1873,0.9462
8,0.2182,0.0675,0.2598,0.2535,0.1763,0.8977


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


In [51]:
#predict_model(blend2_tune);
predict_model(blend2);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2082,0.0638,0.2526,0.2893,0.1712,0.8826


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [52]:
newpred6 = predict_model(blend2_tune, data=data_unseen)
#newpred6 = predict_model(blend2, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2080,0.0654,0.2558,0.2723,0.1726,0.8705


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [27]:
blend3 = blend_models([lgbm_tune, gbr, rf_tune, cat_tune, br_tune, et])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2345,0.0766,0.2768,0.1429,0.1894,1.0148
1,0.2471,0.0849,0.2913,0.0727,0.2013,1.1045
2,0.2353,0.0792,0.2814,0.1286,0.1941,1.0450
3,0.2304,0.0746,0.2732,0.1739,0.1873,0.9775
4,0.2342,0.0762,0.2761,0.1728,0.1884,0.9797
5,0.2205,0.0704,0.2652,0.2124,0.1825,0.9556
6,0.2251,0.0743,0.2726,0.1760,0.1850,0.9464
7,0.2321,0.0758,0.2753,0.1753,0.1874,0.9484
8,0.2176,0.0668,0.2585,0.2611,0.1753,0.8945


In [28]:
blend3_tune = tune_model(blend3)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2342,0.0764,0.2764,0.1454,0.1891,1.0154
1,0.2459,0.0840,0.2898,0.0821,0.2002,1.0944
2,0.2350,0.0791,0.2813,0.1292,0.1939,1.0390
3,0.2297,0.0743,0.2725,0.1777,0.1867,0.9694
4,0.2337,0.0759,0.2756,0.1758,0.1879,0.9720
5,0.2200,0.0701,0.2647,0.2155,0.1820,0.9508
6,0.2250,0.0742,0.2725,0.1770,0.1849,0.9445
7,0.2315,0.0755,0.2748,0.1786,0.1869,0.9449
8,0.2174,0.0667,0.2582,0.2624,0.1752,0.8940


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [54]:
predict_model(blend3_tune);
#predict_model(blend3);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2074,0.0633,0.2516,0.2946,0.1708,0.8846


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [56]:
#newpred7 = predict_model(blend3_tune, data=data_unseen)
newpred7 = predict_model(blend3, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2077,0.0649,0.2547,0.2787,0.1722,0.8776


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [29]:
blend4 = blend_models([cat_tune, lgbm_tune, br_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2354,0.0777,0.2787,0.1315,0.1902,1.0087
1,0.2433,0.0829,0.2879,0.0944,0.1981,1.0536
2,0.2337,0.0796,0.2822,0.1236,0.1943,1.0260
3,0.2280,0.0741,0.2722,0.1797,0.1859,0.9444
4,0.2325,0.0761,0.2759,0.1740,0.1874,0.9431
5,0.2177,0.0695,0.2635,0.2225,0.1803,0.9158
6,0.2229,0.0737,0.2714,0.1832,0.1837,0.9197
7,0.2306,0.0763,0.2762,0.1705,0.1874,0.9273
8,0.2139,0.0659,0.2567,0.2713,0.1739,0.8689


In [30]:
blend4_tune = tune_model(blend4)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2351,0.0781,0.2794,0.1271,0.1904,0.9981
1,0.2432,0.0831,0.2883,0.0921,0.1980,1.0402
2,0.2329,0.0799,0.2827,0.1207,0.1945,1.0142
3,0.2274,0.0738,0.2717,0.1828,0.1853,0.9329
4,0.2311,0.0758,0.2754,0.1769,0.1869,0.9276
5,0.2166,0.0694,0.2635,0.2226,0.1800,0.9024
6,0.2214,0.0733,0.2708,0.1870,0.1831,0.9034
7,0.2299,0.0762,0.2760,0.1713,0.1873,0.9215
8,0.2128,0.0655,0.2559,0.2754,0.1734,0.8596


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [109]:
#predict_model(blend4);
predict_model(blend4_tune);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2037,0.0635,0.2519,0.2929,0.1700,0.8285


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [108]:
newpred8 = predict_model(blend4, data=data_unseen)
#newpred8 = predict_model(blend4_tune, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2056,0.0654,0.2558,0.2727,0.1717,0.8293


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [31]:
blend5 = blend_models([br_tune, et, rf_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2345,0.0764,0.2763,0.1461,0.1891,1.0194
1,0.2454,0.0838,0.2895,0.0840,0.2002,1.1002
2,0.2347,0.0790,0.2810,0.1311,0.1936,1.0290
3,0.2300,0.0743,0.2726,0.1770,0.1867,0.9698
4,0.2332,0.0758,0.2753,0.1772,0.1879,0.9678
5,0.2200,0.0702,0.2650,0.2138,0.1823,0.9585
6,0.2266,0.0748,0.2734,0.1711,0.1857,0.9509
7,0.2309,0.0748,0.2735,0.1861,0.1860,0.9439
8,0.2185,0.0669,0.2587,0.2595,0.1758,0.9029


In [32]:
blend5_tune = tune_model(blend5)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2342,0.0767,0.2769,0.1423,0.1890,1.0001
1,0.2447,0.0836,0.2891,0.0865,0.1994,1.0765
2,0.2330,0.0790,0.2811,0.1305,0.1935,1.0129
3,0.2287,0.0739,0.2718,0.1820,0.1858,0.9544
4,0.2314,0.0755,0.2747,0.1810,0.1871,0.9470
5,0.2184,0.0701,0.2648,0.2154,0.1816,0.9379
6,0.2247,0.0743,0.2726,0.1763,0.1848,0.9312
7,0.2302,0.0750,0.2738,0.1847,0.1860,0.9337
8,0.2164,0.0661,0.2572,0.2683,0.1746,0.8882


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [62]:
predict_model(blend5_tune);
#predict_model(blend5);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2062,0.0633,0.2516,0.2946,0.1705,0.8735


In [64]:
newpred10 = predict_model(blend5_tune, data=data_unseen)
#newpred10 = predict_model(blend5, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2073,0.0655,0.2559,0.2720,0.1726,0.8598


In [86]:
blend6 = blend_models([lgbm_tune, gbr, rf_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2365,0.0776,0.2785,0.1325,0.1908,1.0324
1,0.2507,0.0872,0.2953,0.0472,0.2039,1.1205
2,0.2385,0.0806,0.2839,0.1129,0.1958,1.0692
3,0.2317,0.0753,0.2744,0.1660,0.1883,0.9862
4,0.2361,0.0770,0.2776,0.1639,0.1895,1.0002
5,0.2219,0.0711,0.2666,0.2041,0.1837,0.9638
6,0.2250,0.0743,0.2725,0.1767,0.1851,0.9520
7,0.2340,0.0768,0.2772,0.1644,0.1891,0.9638
8,0.2194,0.0679,0.2606,0.2485,0.1767,0.9007


In [87]:
blend6_tune = tune_model(blend6)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2369,0.0773,0.2780,0.1354,0.1908,1.0427
1,0.2482,0.0852,0.2919,0.0687,0.2016,1.1069
2,0.2380,0.0803,0.2834,0.1163,0.1953,1.0628
3,0.2308,0.0748,0.2736,0.1713,0.1874,0.9762
4,0.2359,0.0767,0.2770,0.1674,0.1891,0.9919
5,0.2214,0.0705,0.2656,0.2106,0.1828,0.9596
6,0.2255,0.0743,0.2726,0.1760,0.1853,0.9546
7,0.2331,0.0762,0.2761,0.1708,0.1882,0.9587
8,0.2194,0.0679,0.2606,0.2489,0.1767,0.9026


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [88]:
predict_model(blend6_tune);
#predict_model(blend6);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2093,0.0637,0.2524,0.2902,0.1718,0.9044


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [89]:
#newpred = predict_model(blend6_tune, data=data_unseen)
newpred = predict_model(blend6, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2091,0.0646,0.2542,0.2812,0.1725,0.9036


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [35]:
blend7 = blend_models([rf_tune, lgbm_tune, gbr, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2361,0.0776,0.2785,0.1327,0.1906,1.0262
1,0.2487,0.0860,0.2932,0.0605,0.2026,1.1142
2,0.2368,0.0798,0.2826,0.1214,0.1950,1.0614
3,0.2315,0.0755,0.2748,0.1642,0.1885,0.9862
4,0.2359,0.0770,0.2775,0.1640,0.1894,0.9926
5,0.2214,0.0707,0.2658,0.2090,0.1830,0.9580
6,0.2251,0.0744,0.2727,0.1755,0.1851,0.9486
7,0.2333,0.0767,0.2769,0.1658,0.1886,0.9546
8,0.2178,0.0674,0.2595,0.2550,0.1759,0.8930


In [36]:
blend7_tune = tune_model(blend7)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2366,0.0774,0.2782,0.1343,0.1907,1.0344
1,0.2456,0.0838,0.2895,0.0840,0.1998,1.0897
2,0.2364,0.0798,0.2825,0.1220,0.1947,1.0532
3,0.2300,0.0749,0.2736,0.1712,0.1874,0.9697
4,0.2354,0.0768,0.2770,0.1669,0.1888,0.9797
5,0.2205,0.0701,0.2648,0.2153,0.1819,0.9484
6,0.2254,0.0744,0.2727,0.1756,0.1851,0.9499
7,0.2324,0.0763,0.2762,0.1702,0.1879,0.9471
8,0.2175,0.0672,0.2593,0.2564,0.1758,0.8928


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [72]:
predict_model(blend7_tune);
#predict_model(blend7);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2077,0.0636,0.2521,0.2919,0.1710,0.8799


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [76]:
newpred = predict_model(blend7_tune, data=data_unseen)
#newpred = predict_model(blend7, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2073,0.0649,0.2548,0.2780,0.1720,0.8710


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [37]:
blend8 = blend_models([rf_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2371,0.0781,0.2794,0.1269,0.1914,1.0396
1,0.2453,0.0838,0.2895,0.0839,0.2004,1.1132
2,0.2351,0.0789,0.2809,0.1314,0.1939,1.0500
3,0.2318,0.0758,0.2753,0.1608,0.1889,0.9864
4,0.2361,0.0772,0.2778,0.1621,0.1895,0.9817
5,0.2209,0.0700,0.2646,0.2163,0.1819,0.9568
6,0.2267,0.0750,0.2739,0.1681,0.1859,0.9518
7,0.2322,0.0761,0.2759,0.1719,0.1874,0.9435
8,0.2173,0.0672,0.2592,0.2567,0.1759,0.8912


In [38]:
blend8_tune = tune_model(blend8)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2374,0.0780,0.2793,0.1278,0.1914,1.0442
1,0.2454,0.0839,0.2896,0.0836,0.2005,1.1151
2,0.2355,0.0790,0.2811,0.1307,0.1939,1.0514
3,0.2318,0.0757,0.2751,0.1619,0.1887,0.9862
4,0.2361,0.0771,0.2777,0.1631,0.1894,0.9834
5,0.2211,0.0700,0.2646,0.2162,0.1820,0.9591
6,0.2269,0.0750,0.2739,0.1683,0.1859,0.9537
7,0.2322,0.0760,0.2757,0.1734,0.1873,0.9456
8,0.2178,0.0674,0.2596,0.2548,0.1761,0.8942


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [78]:
#predict_model(blend8_tune);
predict_model(blend8);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2092,0.0644,0.2537,0.2828,0.1719,0.8845


In [80]:
#newpred = predict_model(blend8_tune, data=data_unseen)
newpred = predict_model(blend8, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2092,0.0661,0.2571,0.2651,0.1733,0.8701


In [39]:
blend9 = blend_models([lgbm_tune, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2371,0.0780,0.2794,0.1273,0.1913,1.0301
1,0.2444,0.0831,0.2884,0.0915,0.1988,1.0750
2,0.2355,0.0798,0.2825,0.1219,0.1947,1.0455
3,0.2296,0.0750,0.2739,0.1694,0.1875,0.9629
4,0.2353,0.0771,0.2776,0.1635,0.1890,0.9687
5,0.2198,0.0700,0.2645,0.2168,0.1815,0.9381
6,0.2255,0.0746,0.2731,0.1730,0.1852,0.9467
7,0.2319,0.0766,0.2767,0.1672,0.1879,0.9375
8,0.2162,0.0668,0.2585,0.2607,0.1753,0.8857


In [40]:
blend9_tune = tune_model(blend9)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2381,0.0781,0.2794,0.1268,0.1915,1.0375
1,0.2442,0.0830,0.2880,0.0936,0.1983,1.0673
2,0.2363,0.0802,0.2832,0.1175,0.1951,1.0467
3,0.2290,0.0747,0.2733,0.1729,0.1869,0.9554
4,0.2352,0.0769,0.2772,0.1657,0.1887,0.9681
5,0.2196,0.0699,0.2644,0.2174,0.1814,0.9371
6,0.2254,0.0744,0.2728,0.1752,0.1851,0.9488
7,0.2318,0.0764,0.2764,0.1692,0.1878,0.9397
8,0.2168,0.0670,0.2589,0.2583,0.1756,0.8898


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [91]:
predict_model(blend9_tune);
#predict_model(blend9);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2065,0.0634,0.2518,0.2934,0.1705,0.8656


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [93]:
newpred = predict_model(blend9_tune, data=data_unseen)
#newpred = predict_model(blend9, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2061,0.0651,0.2551,0.2764,0.1718,0.8541


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [41]:
blend10 = blend_models([rf_tune, lgbm_tune, gbr, cat_tune, br_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2354,0.0773,0.2780,0.1357,0.1900,1.0146
1,0.2467,0.0849,0.2913,0.0725,0.2010,1.0924
2,0.2350,0.0795,0.2819,0.1253,0.1944,1.0457
3,0.2300,0.0745,0.2730,0.1748,0.1870,0.9701
4,0.2339,0.0762,0.2760,0.1733,0.1880,0.9717
5,0.2197,0.0700,0.2647,0.2159,0.1818,0.9403
6,0.2235,0.0738,0.2716,0.1820,0.1841,0.9317
7,0.2322,0.0763,0.2762,0.1700,0.1879,0.9447
8,0.2162,0.0666,0.2581,0.2633,0.1749,0.8817


In [42]:
blend10_tune = tune_model(blend10)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2350,0.0776,0.2785,0.1324,0.1900,1.0046
1,0.2439,0.0834,0.2889,0.0883,0.1989,1.0644
2,0.2333,0.0793,0.2816,0.1273,0.1940,1.0273
3,0.2284,0.0741,0.2722,0.1799,0.1861,0.9515
4,0.2324,0.0759,0.2756,0.1757,0.1873,0.9470
5,0.2179,0.0695,0.2637,0.2216,0.1806,0.9197
6,0.2227,0.0736,0.2713,0.1839,0.1836,0.9172
7,0.2309,0.0762,0.2761,0.1710,0.1874,0.9299
8,0.2139,0.0658,0.2566,0.2717,0.1738,0.8680


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [95]:
predict_model(blend10_tune);
#predict_model(blend10);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2049,0.0635,0.2520,0.2926,0.1702,0.8432


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [97]:
newpred = predict_model(blend10_tune, data=data_unseen)
#newpred = predict_model(blend10, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2060,0.0653,0.2556,0.2737,0.1716,0.8344


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [43]:
blend11 = blend_models([lgbm_tune, gbr])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2360,0.0780,0.2792,0.1281,0.1909,1.0151
1,0.2535,0.0894,0.2990,0.0233,0.2062,1.1183
2,0.2395,0.0816,0.2857,0.1014,0.1971,1.0751
3,0.2321,0.0759,0.2754,0.1600,0.1890,0.9877
4,0.2363,0.0776,0.2786,0.1576,0.1901,1.0047
5,0.2224,0.0720,0.2683,0.1943,0.1848,0.9604
6,0.2238,0.0741,0.2722,0.1784,0.1847,0.9460
7,0.2349,0.0779,0.2791,0.1527,0.1906,0.9667
8,0.2190,0.0679,0.2606,0.2485,0.1765,0.8962


In [44]:
blend11_tune = tune_model(blend11)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2375,0.0774,0.2783,0.1341,0.1908,1.0351
1,0.2475,0.0848,0.2912,0.0733,0.2005,1.0782
2,0.2381,0.0809,0.2845,0.1094,0.1960,1.0595
3,0.2292,0.0744,0.2728,0.1760,0.1866,0.9593
4,0.2353,0.0767,0.2769,0.1680,0.1887,0.9828
5,0.2205,0.0705,0.2656,0.2103,0.1825,0.9458
6,0.2244,0.0740,0.2720,0.1795,0.1848,0.9500
7,0.2329,0.0766,0.2767,0.1669,0.1886,0.9541
8,0.2186,0.0677,0.2601,0.2515,0.1763,0.8980


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [99]:
predict_model(blend11_tune);
#predict_model(blend11);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2074,0.0633,0.2516,0.2948,0.1710,0.8853


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [101]:
newpred = predict_model(blend11_tune, data=data_unseen)
#newpred = predict_model(blend11, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2070,0.0643,0.2536,0.2851,0.1716,0.8828


[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] bagging_freq is set=6, subsample_freq=0 will be ignored. Current value: bagging_freq=6


In [45]:
blend12 = blend_models([gbr, cat_tune])

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2371,0.0802,0.2831,0.1035,0.1929,1.0017
1,0.2548,0.0911,0.3019,0.0042,0.2086,1.1457
2,0.2382,0.0810,0.2846,0.1084,0.1968,1.0734
3,0.2346,0.0784,0.2801,0.1316,0.1926,1.0133
4,0.2384,0.0787,0.2806,0.1455,0.1914,1.0107
5,0.2228,0.0722,0.2687,0.1919,0.1851,0.9639
6,0.2243,0.0749,0.2737,0.1695,0.1852,0.9390
7,0.2351,0.0784,0.2801,0.1467,0.1908,0.9587
8,0.2167,0.0672,0.2591,0.2572,0.1753,0.8816


In [46]:
blend12_tune = tune_model(blend12)

,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.2364,0.0794,0.2819,0.1115,0.1925,1.0119
1,0.2461,0.0848,0.2912,0.0732,0.2013,1.1034
2,0.2343,0.0793,0.2815,0.1278,0.1944,1.0446
3,0.2326,0.0771,0.2777,0.1464,0.1907,0.9900
4,0.2361,0.0784,0.2800,0.1493,0.1904,0.9726
5,0.2205,0.0705,0.2656,0.2105,0.1822,0.9426
6,0.2262,0.0757,0.2752,0.1603,0.1861,0.9401
7,0.2325,0.0775,0.2783,0.1572,0.1886,0.9307
8,0.2148,0.0664,0.2577,0.2652,0.1746,0.8733


Fitting 10 folds for each of 10 candidates, totalling 100 fits


In [103]:
predict_model(blend12_tune);
#predict_model(blend12);

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2069,0.0648,0.2546,0.2777,0.1717,0.8414


In [105]:
#newpred = predict_model(blend12_tune, data=data_unseen)
newpred = predict_model(blend12, data=data_unseen)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Voting Regressor,0.2081,0.0650,0.2550,0.2769,0.1721,0.8704


In [ ]:
save_model(blend8_tune, "indycar_rf_cat_prequaly_model_v1")

In [ ]:
save_model(blend4, "indycar_cat_lgbm_br_postqualy_model_v1")